# Regional LSTM Extended: Step-by-Step Notebook

This notebook is a staged version of `_001_Regional_LSTM_resume_kfold_Extended.py`.

Defaults are set for a small smoke test using `Input/1_regional_hourly_smoke.txt` so you can test data loading, scaling, batching, and one model update without running the full 10-fold experiment.

## 1. Imports and Paths

In [1]:
import os
import pickle
import random
import sys
import time
from pathlib import Path


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

sys.path.append("../_src/_001_aux_functions")
sys.path.append("../_src/_002_readdata")
sys.path.append("../_src/_003_ml_f")

from functions_evaluation import nse
from functions_training import nse_basin_averaged
from utils import Optimizer, create_folder, set_random_seed, upload_to_device, write_report
from camelsh import camelsh as Datasetclass
from mflstm import MFLSTM as modelclass
import update_sample

PROJECT_DIR = Path.cwd()
path_data_default = (PROJECT_DIR / ".." / ".." / "CAMELSH").resolve()
path_data = Path(os.environ.get("CAMELSH_DATA_DIR", path_data_default)).resolve()
print("Project:", PROJECT_DIR)
print("CAMELSH data:", path_data)

Project: /home/emroush/ornl/tva/Examples
CAMELSH data: /home/emroush/ornl/CAMELSH


## 2. Smoke-Test Switches

Keep `SMOKE_TEST = True` while developing. Set it to `False` only when you are ready for the full extended experiment.

In [2]:
SMOKE_TEST = True
RUN_ONE_TRAINING_EPOCH = False
RUN_VALIDATION_STEP = True
RUN_TESTING_STEP = True

if SMOKE_TEST:
    kfold = 1
    path_entities = "Input/1_regional_hourly.txt"
    test_path_entities = "Input/1_regional_hourly_ungauged.txt"
    experiment_name_prefix = "001_Regional_model_extended_smoke"
else:
    kfold = 10
    path_entities = "Input/1_regional_hourly.txt"
    experiment_name_prefix = "001_Regional_model_extendedn"

entities_ids = np.loadtxt(path_entities, dtype="str").tolist()
entities_ids = [entities_ids] if isinstance(entities_ids, str) else entities_ids
ungauged_entities_ids = np.loadtxt(test_path_entities, dtype="str").tolist()
print(f"Loaded {len(entities_ids)} basin IDs from {path_entities}")
print(entities_ids[:10])
RUN_SMOKE_FULL_EXPERIMENT = True
_debug_batch_limit = os.environ.get("MAX_SMOKE_TRAIN_BATCHES_PER_EPOCH", "").strip()
MAX_SMOKE_TRAIN_BATCHES_PER_EPOCH = int(_debug_batch_limit) if _debug_batch_limit else None
# Leave unset for full smoke training; set env var MAX_SMOKE_TRAIN_BATCHES_PER_EPOCH=50 for quick debugging.

Loaded 2 basin IDs from Input/1_regional_hourly.txt
['01123000', '01137500']


## 3. Inputs and Model Configuration

In [3]:
dynamic_input = {
    "1D": ["CAPE", "CRainf_frac", "LWdown", "PotEvap", "PSurf", "Qair", "Rainf", "SWdown", "Tair", "Wind_E", "Wind_N"],
    "1h": ["CAPE", "CRainf_frac", "LWdown", "PotEvap", "PSurf", "Qair", "Rainf", "SWdown", "Tair", "Wind_E", "Wind_N"],
}
target = ["Q_camelsh_obs_norm"]
forcing = ["nldas_hourly"]
static_input = [
    "p_mean", "pet_mean", "aridity_index", "p_seasonality", "frac_snow",
    "high_prec_freq", "high_prec_dur", "low_prec_freq", "low_prec_dur",
    "ele_mt_sav", "slp_dg_uav", "ria_ha_usu", "run_mm_syr", "gwt_cm_sav",
    "cly_pc_uav", "slt_pc_uav", "snd_pc_uav", "kar_pc_use", "prm_pc_use",
    "pac_pc_use", "crp_pc_use", "for_pc_use", "urb_pc_use", "DRAIN_SQKM"
]

if SMOKE_TEST:
    training_period = ["1987-01-01 00:00:00", "2009-12-31 23:00:00"]
    validation_period = ["2010-01-01 00:00:00", "2015-12-31 23:00:00"]
    testing_period = ["2016-01-01 00:00:00", "2022-12-31 23:00:00"]
else:
    training_period = ["1981-01-01 00:00:00", "2007-12-31 23:00:00"]
    validation_period = ["2008-01-01 00:00:00", "2011-12-31 23:00:00"]
    testing_period = ["1981-01-01 00:00:00", "2022-12-31 23:00:00"]

lookback_window = 0
model_configuration = {
    "n_dynamic_channels_lstm": 10,
    "no_of_layers": 1,
    "seq_length": 365 * 24,
    "custom_freq_processing": {
        "1D": {"n_steps": 351, "freq_factor": 24},
        "1h": {"n_steps": (365 - 351) * 24, "freq_factor": 1},
    },
    "predict_last_n": 1,
    "unique_prediction_blocks": True,
    "dynamic_embeddings": True,
    "hidden_size": 32 if SMOKE_TEST else 128,
    "batch_size_training": 16 if SMOKE_TEST else 128,
    "batch_size_evaluation": 128 if SMOKE_TEST else 1024,
    "no_of_epochs": 10 if SMOKE_TEST else 30,
    "dropout_rate": 0.4,
    "learning_rate": {1: 5e-4, 10: 1e-4, 25: 1e-5},
    "set_forget_gate": 3,
    "validate_every": 1 if SMOKE_TEST else 4,
    "validate_n_random_basins": 1 if SMOKE_TEST else -1,
}

seed = 110
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}

## 4. Derived Configuration and Device

In [4]:
if isinstance(dynamic_input, list):
    model_configuration["dynamic_input_size"] = len(dynamic_input)
elif isinstance(dynamic_input, dict):
    model_configuration["dynamic_input_size"] = {key: len(value) for key, value in dynamic_input.items()}

model_configuration["input_size_lstm"] = model_configuration["n_dynamic_channels_lstm"] + len(static_input)
if model_configuration.get("custom_freq_processing") and not model_configuration.get("dynamic_embeddings"):
    model_configuration["input_size_lstm"] += 1

if not model_configuration.get("predict_last_n"):
    model_configuration["predict_last_n"] = 1

if model_configuration.get("predict_last_n", 1) > 1 and not model_configuration.get("unique_prediction_blocks"):
    model_configuration["predict_last_n_evaluation"] = 1
else:
    model_configuration["predict_last_n_evaluation"] = model_configuration.get("predict_last_n", 1)

device = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)
print("Torch:", torch.__version__)
print("MPS built:", torch.backends.mps.is_built(), "available:", torch.backends.mps.is_available())

Using device: cuda:0
Torch: 2.12.0+cu130
MPS built: False available: False


## 5. Build Fold Split

In [5]:
if SMOKE_TEST:
    cv_folds = [{"train": entities_ids[:2], "test": entities_ids[2:]}]
else:
    kfold_file = "basin_kfold_10_extended.pkl"
    if os.path.exists(kfold_file):
        print("Loading existing K-fold splits...")
        with open(kfold_file, "rb") as f:
            cv_folds = pickle.load(f)
    else:
        print("Creating new K-fold splits...")
        shuffled_ids = entities_ids.copy()
        np.random.seed(42)
        np.random.shuffle(shuffled_ids)
        folds = [list(f) for f in np.array_split(shuffled_ids, 10)]
        cv_folds = []
        for i in range(10):
            cv_folds.append({
                "train": [b for j, f in enumerate(folds) if j != i for b in f],
                "test": folds[i],
            })
        with open(kfold_file, "wb") as f:
            pickle.dump(cv_folds, f)
        print("K-fold splits saved.")

fold_idx = 0
train_basins = cv_folds[fold_idx]["train"]
test_basins = cv_folds[fold_idx]["test"]
test_basins = ungauged_entities_ids
experiment_name = f"{experiment_name_prefix}_k={fold_idx + 1}"
path_save_folder = Path("Results") / experiment_name
create_folder(str(path_save_folder))

print("Train basins:", train_basins)
print("Test basins:", test_basins)
print("Output folder:", path_save_folder)

Folder 'Results/001_Regional_model_extended_smoke_k=1' already exists.
Train basins: ['01123000', '01137500']
Test basins: ['01123000', '01137500']
Output folder: Results/001_Regional_model_extended_smoke_k=1


## 6. Build Training Dataset

This is the slow loading/preprocessing step. The smoke-test version should complete much faster than the full extended run.

In [6]:
if not (path_data / "attributes").exists():
    raise FileNotFoundError(f"CAMELSH data folder not found or incomplete: {path_data}")

train_entity_file = path_save_folder / "train_entities.txt"
np.savetxt(train_entity_file, train_basins, fmt="%s")

start = time.time()
training_dataset = Datasetclass(
    dynamic_input=dynamic_input,
    forcing=forcing,
    target=target,
    sequence_length=model_configuration["seq_length"],
    time_period=training_period,
    path_data=str(path_data),
    path_entities=str(train_entity_file),
    check_NaN=True,
    predict_last_n=model_configuration["predict_last_n"],
    static_input=static_input,
    custom_freq_processing=model_configuration["custom_freq_processing"],
    dynamic_embedding=model_configuration["dynamic_embeddings"],
    unique_prediction_blocks=model_configuration["unique_prediction_blocks"],
    lookback_window=lookback_window,
)
print(f"Training dataset built in {time.time() - start:.1f}s")
print("Valid samples:", len(training_dataset))
print("Basins with sequence data:", list(training_dataset.sequence_data.keys()))

Training dataset built in 40.3s
Valid samples: 132095
Basins with sequence data: ['01123000']


## 7. Scale Data and Build DataLoader

In [7]:
training_dataset.calculate_basin_std()
training_dataset.calculate_global_statistics(path_save_scaler=str(path_save_folder))
scaler = training_dataset.scaler
training_dataset.standardize_data()

train_loader = DataLoader(
    dataset=training_dataset,
    batch_size=model_configuration["batch_size_training"],
    shuffle=True,
    drop_last=True,
    collate_fn=training_dataset.collate_fn,
)
print("Number of batches in training:", len(train_loader))

Number of batches in training: 8255


## 8. Inspect One Batch

In [8]:
sample = next(iter(train_loader))
for key, value in sample.items():
    if hasattr(value, "shape"):
        print(key, value.shape)
    else:
        print(key, type(value))

x_d_1D torch.Size([16, 351, 11])
x_d_1h torch.Size([16, 336, 11])
x_s torch.Size([16, 24])
y_obs torch.Size([16, 1, 1])
basin_std torch.Size([16, 1, 1])
basin (16,)
date (16, 1)


## 9. Model Setup and One Forward Pass

In [9]:
set_random_seed(seed)
model = modelclass(model_configuration=model_configuration).to(device)
optimizer = Optimizer(model=model, model_configuration=model_configuration)
model.lstm.bias_hh_l0.data[model_configuration["hidden_size"]:2 * model_configuration["hidden_size"]] = model_configuration["set_forget_gate"]

sample_device = upload_to_device(sample, device)
with torch.no_grad():
    pred = model(sample_device)
print("Prediction shape:", pred["y_sim"].shape)

Prediction shape: torch.Size([16, 1, 1])


## 10. Optional One-Epoch Training Smoke Test

In [10]:
if RUN_ONE_TRAINING_EPOCH:
    model.train()
    epoch_start_time = time.time()
    total_loss = []
    for idx, sample in enumerate(train_loader):
        sample = upload_to_device(sample, device)
        optimizer.optimizer.zero_grad()
        pred = model(sample)
        loss = nse_basin_averaged(
            y_sim=pred["y_sim"],
            y_obs=sample["y_obs"],
            per_basin_target_std=sample["basin_std"],
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.optimizer.step()
        total_loss.append(loss.item())
        if SMOKE_TEST and idx >= 2:
            break
    print(f"Smoke training loss: {np.mean(total_loss):.3f}; time: {time.time() - epoch_start_time:.1f}s")
else:
    print("Skipping training smoke test")

Skipping training smoke test


## 11. Optional Validation Dataset

In [11]:
validation_dataset = {}

if RUN_VALIDATION_STEP:
    validation_basins = train_basins[:model_configuration["validate_n_random_basins"]] if SMOKE_TEST else train_basins
    for entity in validation_basins:
        dataset = Datasetclass(
            dynamic_input=dynamic_input,
            forcing=forcing,
            target=target,
            sequence_length=model_configuration["seq_length"],
            time_period=validation_period,
            path_data=str(path_data),
            entity=entity,
            check_NaN=False,
            predict_last_n=model_configuration["predict_last_n"],
            static_input=static_input,
            custom_freq_processing=model_configuration["custom_freq_processing"],
            dynamic_embedding=model_configuration["dynamic_embeddings"],
            unique_prediction_blocks=model_configuration["unique_prediction_blocks"],
            lookback_window=lookback_window,
        )
        dataset.scaler = training_dataset.scaler
        dataset.standardize_data(standardize_output=False)
        validation_dataset[entity] = dataset
    print("Validation datasets:", list(validation_dataset.keys()))
else:
    print("Skipping validation dataset build")

Validation datasets: ['01123000']


## 12. Smoke 10-Epoch Training With Validation

This cell trains the small smoke model for `model_configuration["no_of_epochs"]` epochs. With the current smoke defaults, that is 10 epochs. Set `MAX_SMOKE_TRAIN_BATCHES_PER_EPOCH` to a small integer if you want a faster debug pass.

In [12]:
if RUN_SMOKE_FULL_EXPERIMENT:
    if RUN_VALIDATION_STEP and not validation_dataset:
        raise RuntimeError("RUN_VALIDATION_STEP is True, but validation_dataset is empty. Run the validation dataset cell first.")

    training_time = time.time()
    best_validation = -np.inf
    best_epoch = None

    for epoch in range(1, model_configuration["no_of_epochs"] + 1):
        epoch_start_time = time.time()
        total_loss = []
        model.train()

        for idx, sample in enumerate(train_loader, start=1):
            sample = upload_to_device(sample, device)
            optimizer.optimizer.zero_grad()
            pred = model(sample)
            loss = nse_basin_averaged(
                y_sim=pred["y_sim"],
                y_obs=sample["y_obs"],
                per_basin_target_std=sample["basin_std"],
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
            optimizer.optimizer.step()
            total_loss.append(loss.item())

            if MAX_SMOKE_TRAIN_BATCHES_PER_EPOCH is not None and idx >= MAX_SMOKE_TRAIN_BATCHES_PER_EPOCH:
                break

        report = f'Epoch: {epoch:<2} | Loss training: {np.mean(total_loss):.3f}'

        if RUN_VALIDATION_STEP and epoch % model_configuration["validate_every"] == 0:
            model.eval()
            validation_results = {}
            with torch.no_grad():
                for basin, dataset in validation_dataset.items():
                    loader = DataLoader(
                        dataset=dataset,
                        batch_size=model_configuration["batch_size_evaluation"],
                        shuffle=False,
                        drop_last=False,
                        collate_fn=dataset.collate_fn,
                    )
                    frames = []
                    for sample in loader:
                        sample = upload_to_device(sample, device)
                        pred = model(sample)
                        y_sim = pred["y_sim"] * dataset.scaler["y_std"].to(device) + dataset.scaler["y_mean"].to(device)
                        frames.append(pd.DataFrame({
                            "y_obs": sample["y_obs"].flatten().cpu().detach(),
                            "y_sim": y_sim[:, -model_configuration["predict_last_n"]:, :].flatten().cpu().detach(),
                        }, index=pd.to_datetime(sample["date"].flatten())))
                    validation_results[basin] = pd.concat(frames, axis=0) if frames else pd.DataFrame(columns=["y_obs", "y_sim"])

            loss_validation = nse(df_results=validation_results)
            report += f' | NSE validation: {loss_validation:.3f}'
            if loss_validation > best_validation:
                best_validation = loss_validation
                best_epoch = epoch
                torch.save(model.state_dict(), path_save_folder / "best_epoch_smoke.pt")

        torch.save(model.state_dict(), path_save_folder / f"epoch_{epoch}")
        torch.save(optimizer.optimizer.state_dict(), path_save_folder / f"optimizer_epoch_{epoch}.pt")
        report += f' | Epoch time: {time.time() - epoch_start_time:.1f}s | LR: {optimizer.optimizer.param_groups[0]["lr"]:.5f}'
        print(report)
        write_report(file_path=str(path_save_folder / "run_progress.txt"), text=report)
        optimizer.update_optimizer_lr(epoch=epoch)

    print(f"Total smoke training time: {time.time() - training_time:.1f}s")
    if best_epoch is not None:
        print(f"Best validation NSE: {best_validation:.3f} at epoch {best_epoch}")
else:
    print("Smoke full experiment disabled")

Epoch: 1  | Loss training: 0.052 | NSE validation: 0.665 | Epoch time: 217.7s | LR: 0.00050
Epoch: 2  | Loss training: 0.030 | NSE validation: 0.756 | Epoch time: 211.3s | LR: 0.00050
Epoch: 3  | Loss training: 0.025 | NSE validation: 0.757 | Epoch time: 214.6s | LR: 0.00050
Epoch: 4  | Loss training: 0.021 | NSE validation: 0.761 | Epoch time: 223.3s | LR: 0.00050
Epoch: 5  | Loss training: 0.018 | NSE validation: 0.744 | Epoch time: 220.7s | LR: 0.00050
Epoch: 6  | Loss training: 0.016 | NSE validation: 0.729 | Epoch time: 210.9s | LR: 0.00050
Epoch: 7  | Loss training: 0.015 | NSE validation: 0.700 | Epoch time: 207.9s | LR: 0.00050
Epoch: 8  | Loss training: 0.014 | NSE validation: 0.712 | Epoch time: 215.7s | LR: 0.00050
Epoch: 9  | Loss training: 0.013 | NSE validation: 0.697 | Epoch time: 208.5s | LR: 0.00050
Epoch: 10 | Loss training: 0.013 | NSE validation: 0.708 | Epoch time: 207.5s | LR: 0.00050
Total smoke training time: 2138.3s
Best validation NSE: 0.761 at epoch 4


## 13. Smoke Test Set Evaluation

This evaluates the held-out smoke basin(s), saves `test_results_smoke.pickle`, and writes `NSE_testing_smoke.csv`.

In [13]:

if RUN_TESTING_STEP:
    testing_dataset = {}
    for entity in test_basins:
        dataset = Datasetclass(
            dynamic_input=dynamic_input,
            forcing=forcing,
            target=target,
            sequence_length=model_configuration["seq_length"],
            time_period=testing_period,
            path_data=str(path_data),
            entity=entity,
            check_NaN=False,
            predict_last_n=model_configuration["predict_last_n_evaluation"],
            static_input=static_input,
            custom_freq_processing=model_configuration["custom_freq_processing"],
            dynamic_embedding=model_configuration["dynamic_embeddings"],
            unique_prediction_blocks=model_configuration["unique_prediction_blocks"],
            lookback_window=lookback_window,
        )
        dataset.scaler = scaler
        dataset.standardize_data(standardize_output=False)
        testing_dataset[entity] = dataset

    model.eval()
    test_results = {}
    with torch.no_grad():
        for basin, dataset in testing_dataset.items():
            loader = DataLoader(
                dataset=dataset,
                batch_size=model_configuration["batch_size_evaluation"],
                shuffle=False,
                drop_last=False,
                collate_fn=dataset.collate_fn,
            )
            frames = []
            for sample in loader:
                sample = upload_to_device(sample, device)
                pred = model(sample)
                y_sim = pred["y_sim"] * dataset.scaler["y_std"].to(device) + dataset.scaler["y_mean"].to(device)
                frames.append(pd.DataFrame({
                    "y_obs": sample["y_obs"].flatten().cpu().detach(),
                    "y_sim": y_sim[:, -model_configuration["predict_last_n_evaluation"]:, :].flatten().cpu().detach(),
                }, index=pd.to_datetime(sample["date"].flatten())))
            test_results[basin] = pd.concat(frames, axis=0) if frames else pd.DataFrame(columns=["y_obs", "y_sim"])

    with open(path_save_folder / "test_results_smoke.pickle", "wb") as f:
        pickle.dump(test_results, f)

    loss_testing = nse(df_results=test_results, average=False)
    df_NSE = pd.DataFrame({"basin_id": list(testing_dataset.keys()), "NSE": np.round(loss_testing, 3)}).set_index("basin_id")
    df_NSE.to_csv(path_save_folder / "NSE_testing_smoke.csv", index=True, header=True)
    display(df_NSE)
else:
    print("Skipping smoke test evaluation")

,NSE
basin_id,
01123000,0.622
01137500,NaN
